**Create Dataset**

In [2]:
TRAINING_DATA_PATH = "training_data/synth_train_data.npz"

In [4]:
import librosa
import torch
import numpy as np
from helper_functions import AudioRecordingDataset, generate_dataset

In [ ]:
X, Y, HV = generate_dataset()
np.savez(file = TRAINING_DATA_PATH, X = X,Y = Y, hv = HV)

**Load Torch Dataset/Dataloader**

In [6]:
from torch.utils.data import DataLoader, Subset
import torch.nn as nn
import torch.optim as optim

data = AudioRecordingDataset(TRAINING_DATA_PATH)
val_mask = (data.hv == 0.25)
val_idx = np.where(val_mask)[0]
train_idx = np.where(~val_mask)[0]


train_X = data.X[train_idx]
data.mean = train_X.mean(axis = 0)
data.std = train_X.std(axis = 0) + 1e-8 #for 0 values

print(f"data_mean: {data.mean.shape}| data_std: {data.std.shape}")

train_data = Subset(data, train_idx)
val_data = Subset(data, val_idx)

print(f"train {len(train_data)} | val {len(val_data)}")

train_loader = DataLoader(
    dataset = train_data,
    batch_size = 32,
    shuffle = True
)
val_loader = DataLoader(
    dataset = val_data,
    batch_size = 32,
    shuffle = True
)

# batch size x feature_size (64, 84)
model = nn.Sequential(
    nn.Linear(84, 125), 
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(125, 88) # 88 = num of valid midi_notes
)
# batch size * output_dim (64, 88)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

data_mean: (84,)| data_std: (84,)
train 1760 | val 440


In [7]:
num_epochs = 35

for epoch in range(num_epochs):
    model.train() 
    total_loss = 0
    for batch_idx, (batch_X, batch_Y) in enumerate(train_loader):
        logits = model(batch_X)
        loss = criterion(logits, batch_Y)
        #back pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0,0,0
    with torch.no_grad():
        for batch_X, batch_Y in val_loader:
            logits = model(batch_X)
            val_loss += criterion(logits, batch_Y).item()
            total += batch_Y.size(0)
            correct += (logits.argmax(dim=1) == batch_Y).sum().item()

    print(f"Epoch {epoch+1:3d} | train {total_loss/len(train_loader):.3f} "
          f"| val {val_loss/len(val_loader):.3f}| acc {correct/total:.3f}")


Epoch   1 | train 4.317 | val 3.976| acc 0.352
Epoch   2 | train 3.684 | val 3.230| acc 0.532
Epoch   3 | train 2.937 | val 2.478| acc 0.750
Epoch   4 | train 2.296 | val 1.894| acc 0.820
Epoch   5 | train 1.814 | val 1.481| acc 0.873
Epoch   6 | train 1.495 | val 1.221| acc 0.889
Epoch   7 | train 1.261 | val 1.021| acc 0.907
Epoch   8 | train 1.095 | val 0.871| acc 0.932
Epoch   9 | train 0.971 | val 0.762| acc 0.936
Epoch  10 | train 0.843 | val 0.661| acc 0.957
Epoch  11 | train 0.761 | val 0.593| acc 0.959
Epoch  12 | train 0.687 | val 0.533| acc 0.973
Epoch  13 | train 0.634 | val 0.485| acc 0.973
Epoch  14 | train 0.574 | val 0.439| acc 0.977
Epoch  15 | train 0.526 | val 0.402| acc 0.973
Epoch  16 | train 0.480 | val 0.369| acc 0.986
Epoch  17 | train 0.465 | val 0.342| acc 0.977
Epoch  18 | train 0.417 | val 0.316| acc 0.995
Epoch  19 | train 0.396 | val 0.293| acc 0.993
Epoch  20 | train 0.367 | val 0.275| acc 0.989
Epoch  21 | train 0.354 | val 0.257| acc 0.986
Epoch  22 | t

In [ ]:
# Save the Model
checkpoint = {
    'model_state': model.state_dict(),
    'mean' : torch.tensor(data.mean, dtype=torch.float32),
    'std' : torch.tensor(data.std, dtype=torch.float32)
}

torch.save(checkpoint, 'model_bundle.pth')

**Test Validation**

Load Model

In [ ]:
#Generate Validate dataset

VALIDATE_DATA_PATH = "validation_data/validate_data.npz"

X, Y, hv = load_dataset()
np.savez(file = VALIDATE_DATA_PATH, X = X,Y = Y, hv = HV)

In [ ]:
checkpoint = torch.load('model_bundle.pth')

# Define model Architecture
model = nn.Sequential(
    nn.Linear(84, 125), 
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(125, 88) # 88 = num of valid midi_notes
)

data = AudioRecordingDataset(VALIDATE_DATA_PATH)

model.load_state_dict(checkpoint['model_state'])
model.eval()

data.mean = checkpoint["mean"]
data.std = checkpoint["std"]



TODO: Add commenting to all refactored functions (params, returns, brief)

In [ ]:
#%----- Bin # to Note checker-----

# Verify that peak bin corresponds to A4 (440 Hz)
fmin = librosa.note_to_hz('C1')  # 32.7 Hz #lowest bin frequency
freqs = librosa.cqt_frequencies(n_bins=84, fmin=fmin, bins_per_octave=12)
peak_bin = 45 # CHANGE THIS VALUE FOR CHECKER
peak_freq = freqs[peak_bin]
a4_freq = librosa.note_to_hz('A4')  # 440 Hz

print(f"Peak bin: {peak_bin}")
print(f"Peak frequency: {peak_freq:.2f} Hz")
print(f"A4 frequency: {a4_freq:.2f} Hz")
print(f"Difference: {abs(peak_freq - a4_freq):.2f} Hz")
print(f"✓ Phase 1 verified: Peak bin corresponds to A4" if abs(peak_freq - a4_freq) < 5 else "✗ Peak bin does NOT match A4")



In [ ]:
#%----General Plotting-----$#
import librosa.display
import matplotlib.pyplot as plt

# Create subplots: CQT (freq vs amp) and waveform (time vs amp)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Plot CQT spectrogram (frequency vs amplitude)
img = librosa.display.specshow(amp_to_db(cqt, ref=np.max(cqt)), sr=sr, x_axis='time', y_axis='cqt_hz', ax=ax1)
ax1.set_title('CQT Spectrogram (Frequency vs Amplitude)')
fig.colorbar(img, ax=ax1, format='%+2.0f dB')

# Plot waveform (time vs amplitude)
librosa.display.waveshow(y, sr=sr, ax=ax2)
ax2.set_title('Waveform (Time vs Amplitude)')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

In [ ]:
#%----CQT Line Graph
# Plot CQT as line graph (magnitude vs frequency)
fig, ax = plt.subplots(figsize=(12, 6))

# Average CQT magnitude across time to get overall frequency content
cqt_mag = np.abs(cqt)
cqt_mean = np.mean(cqt_mag, axis=1)

# Get frequency values for CQT bins
freqs = librosa.cqt_frequencies(n_bins=cqt.shape[0], fmin=librosa.note_to_hz('C1'), bins_per_octave=12)

# Plot as line graph
ax.plot(freqs, librosa.amplitude_to_db(cqt_mean, ref=np.max(cqt_mean)), linewidth=1.5)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Amplitude (dB)')
ax.set_title('CQT Line Graph (Frequency vs Amplitude)')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()